In [6]:
# import pandas as pd
# import numpy as np
# import os
# import  glob

# !pip install duckdb

# **PHASE 1 — DATA FOUNDATION**
- Let get all trade summary file from local folder
- merge them into single master csv data file
- normalize column names for better understanding and model standards

Company Name| Symbol| Share Volume| Trade Volume| Previous Close (Rs.)| Open (Rs.)| High (Rs.)|Low (Rs.)| **Last Trade (Rs.)| Change (Rs.)| Change (%)

##### Before modeling, you should now write a schema mapper that converts this raw exchange format → standardized ML format
Last Trade	    >  close<br>
Previous Close	>  prev_close<br>
Share Volume	>  volume<br>
Trade Volume	>  trades<br>

### Scan "C:\Users\Lapmart\Downloads\CSE" folder and crate master data set

## Data handling with pandas
- slower when having millions of data

In [7]:
import pandas as pd
import os
import glob
from datetime import datetime


# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE"
OUTPUT = "MASTER_CSE_DATA.csv"

os.makedirs("log", exist_ok=True)
LOG_FILE = LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")


# ================ HELPERS =============

def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        log(f"### Error exctracting name from {filename}")
        return None

# Column mapping
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}


# ================ LOAD FILES ========================

files = glob.glob(os.path.join(FOLDER, "*.csv"))

log("=== LOADER START USING PANDAS FOR DATA ===")
log(f"Total files found: {len(files)}")

valid_frames = []
valid_dates = []

for file in files:
    name = os.path.basename(file)

     # ---- check filename date
    date = extract_date(name)
    if date is None:
        log(f"INVALID DATE FORMAT → skipped: {name}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
    except Exception as e:
        log(f"### READ ERROR → {name} → {e}")
        continue
        
    # ---- empty file check
    if df.empty:
        log(f"EMPTY FILE → skipped: {name}")
        continue

    # ---- normalize column names
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    # ---- add date column
    df["date"] = date

    valid_frames.append(df)
    valid_dates.append(date)


# ================ MERGE =======================

if not valid_frames:
    log("NO VALID FILES FOUND")
    exit()
    
# combine all files
master = pd.concat(valid_frames, ignore_index=True)


# ================ DUPLICATE CHECK =====================

dupes = master.duplicated(subset=["symbol","date"])
dup_count = dupes.sum()

if dup_count > 0:
    log(f"DUPLICATES FOUND: {dup_count} rows removed")
    master = master[~dupes]


# ================ SORT ========================

master = master.sort_values(["symbol", "date"])


# # ================ MISSING DATE REPORT ==================

# valid_dates = sorted(set(valid_dates))
# missing_days = []

# for i in range(len(valid_dates)-1):
#     diff = (valid_dates[i+1] - valid_dates[i]).days
#     if diff > 1:
#         missing_days.append((valid_dates[i], valid_dates[i+1]))

# if missing_days:
#     log("MISSING DATE GAPS:")
#     for g in missing_days:
#         log(f"{g[0].date()} → {g[1].date()}")


# ================ SAVE =======================
        
master.to_csv(OUTPUT, index=False)

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED ===")
print("DONE — Master dataset created using pandas")


=== LOADER START USING PANDAS FOR DATA ===
Total files found: 54
DUPLICATES FOUND: 293 rows removed
MASTER DATASET CREATED → MASTER_CSE_DATA.csv
=== LOADER FINISHED ===
DONE — Master dataset created using pandas


## Data Hndling with DuckDB

### Why Data Engineers Prefer DuckDB
- Real data pipelines often use it for:
- stock market analysis
- log processing
- machine learning datasets
- financial backtesting
- Because it handles millions to billions of rows locally.

In [9]:
import pandas as pd
import os
import glob # finds files/folders matching patterns like wildcards in Python
from datetime import datetime
import duckdb



# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE" # my local stored folder path
OUTPUT = "MASTER_CSE_DATA.csv" # Name of the master data file

os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")



# ================ HELPERS =================

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

# Method to extract date from file name in format like 260210_tradesummary.csv.
def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        return None

# Standardize column names, helper map for mapping existing unusual column names into standard names 
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}



# ================= CONNECT DUCKDB =================

con = duckdb.connect(database="cse_data.db")  # Create/connect DB
log("Creating DuckDB table 'stocks' ...")

con.execute("""
CREATE TABLE IF NOT EXISTS stocks (
    company TEXT,
    symbol TEXT,
    low DOUBLE,
    close DOUBLE,
    prev_close DOUBLE,
    open DOUBLE,
    high DOUBLE,
    volume DOUBLE,
    trades DOUBLE,
    change DOUBLE,
    change_percentage DOUBLE,
    date DATE
)
""")



# ================= LOAD CSVs DIRECTLY =================

log("=== LOADER START WITH DUCKDB FOR DATA HANDLING ===")

files = glob.glob(os.path.join(FOLDER, "*.csv"))
log(f"Total files found: {len(files)}")

for file in files:
    filename = os.path.basename(file) # extract base name
    date = extract_date(filename) # get date from filename though function extract_date()
    
    if date is None:
        log(f"### INVALID DATE FORMAT → skipped: {filename}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
        if df.empty:
            log(f"EMPTY FILE → skipped: {filename}")
            continue
    except Exception as e:
        log(f"READ ERROR → {filename} → {e}")
        continue
        
    # ---- Standardize column names, We assign a new list to df.columns to update the column names after processing them
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True) # Renames columns based on COLUMN_MAP.


    # ======== DATA QUALITY REPORT =========
    
    no_of_total_rows = len(df)
    no_of_missing_symbol = df["symbol"].isna().sum() if "symbol" in df.columns else total_rows
    no_of_missing_close = df["close"].isna().sum() if "close" in df.columns else total_rows

    numeric_cols = ["low","close","prev_close","open","high","volume","trades","change","change_percentage"]
    invalid_numeric = {}

    for col in numeric_cols:
        if col in df.columns:
            invalid_numeric[col] = pd.to_numeric(df[col], errors="coerce").isna().sum()
            
    if(no_of_missing_symbol > 0 or no_of_missing_close > 0):
        log("\n============== DATA QUALITY REPORT ==============")
        log(f"FILE: {filename}")
        log(f"Total rows: {no_of_total_rows}")
        log(f"Missing symbols: {no_of_missing_symbol}")
        log(f"Missing close prices: {no_of_missing_close}")
        log("============== ENF OF DATA QUALITY REPORT ==============\n")

    for col, count in invalid_numeric.items():
        if count > 0:
            log(f"Invalid numeric values in {col}: {count}")

    # check negative prices
    if "close" in df.columns:
        neg_price = (pd.to_numeric(df["close"], errors="coerce") <= 0).sum()
        if neg_price > 0:
            log(f"Invalid negative prices: {neg_price}")

    
    # ================= CLEAN DATA =================

    no_of_rows_before_clean = len(df)

    # remove rows where symbol missing BEFORE casting to string
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        
    # strip spaces
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        df["symbol"] = df["symbol"].astype(str).str.strip()

    if "company" in df.columns:
        df["company"] = df["company"].astype(str).str.strip()
    
    # convert numeric columns
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # remove rows where close price missing
    if "close" in df.columns:
        df = df[df["close"].notna()]

    # remove impossible values
    if "close" in df.columns:
        df = df[df["close"] > 0]

    if "volume" in df.columns:
        df = df[df["volume"] >= 0]

    # fix percentage anomalies
    if "change_percentage" in df.columns:
        df.loc[(df["change_percentage"] < -100) | (df["change_percentage"] > 1000), "change_percentage"] = None

    # drop fully empty rows
    df = df.dropna(how="all")

    no_of_rows_after_clean = len(df)
    
    if(no_of_rows_before_clean > no_of_rows_after_clean):
        log("\n========== DATA CLEAN REPORT ========")
        log(f"Some rows removed, File : {filename} before cleaning → {no_of_rows_before_clean} rows, After {no_of_rows_after_clean} rows\n")

    if len(df) == 0:
        log(f"ALL ROWS REMOVED AFTER CLEANING → skipped {filename}")
        continue
    
    # ---- add date column
    df["date"] = date

    # ------- Insert directly into DuckDB 'stocks' table
    con.register("temp_df", df)
    con.execute("INSERT INTO stocks SELECT * FROM temp_df")
    # log(f"Inserted {len(df)} rows from {filename}")

log("All CSVs inserted into DuckDB.")



# ================  REMOVE DUPLICATES =====================
# Removes duplicate rows for the same symbol and date
# numbers duplicate rows starting from 1
# WHERE rn = 1 → keeps only the first row of each symbol/date combination

log("Removing duplicates...")

# Creates a table named stocks_clean
# If it already exists → deletes old one and recreates it
# add ne column rn, assign occurnce counnt for each record only keeep occurence number 1 others will be delted
# 
con.execute("""
CREATE OR REPLACE TABLE stocks_clean AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY symbol, date
               ORDER BY date
           ) AS rn
    FROM stocks
)
WHERE rn = 1
""")



# ================ SORT ========================

log("Sorting dataset...")

# Sorts the cleaned dataset by symbol then date
con.execute("""
CREATE OR REPLACE TABLE stocks_sorted AS
SELECT *
FROM stocks_clean
ORDER BY symbol, date
""")



# ================ CREATE INDEX ========================

log("Creating composite index on symbol+date for speed...")

# Adds a composite index on (symbol, date)
# Speeds up all queries that filter by symbol, date, or both
con.execute("""
CREATE INDEX IF NOT EXISTS idx_symbol_date ON stocks_sorted(symbol, date)
""")



# ================ MISSING DATE REPORT ==================
# TODO : if needed



# ================ EXPORT =======================

log("Exporting final CSV...")

con.execute(f"""
COPY stocks_sorted TO '{OUTPUT}' (HEADER, DELIMITER ',')
""")

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED USING DUCKDB ===")
print("DONE — Master dataset created")

Creating DuckDB table 'stocks' ...
=== LOADER START WITH DUCKDB FOR DATA HANDLING ===
Total files found: 54

============== DATA QUALITY REPORT ==============
FILE: 260228_trade-summary-test-dup.csv
Total rows: 293
Missing symbols: 1
Missing close prices: 1
============== ENF OF DATA QUALITY REPORT ==============

Invalid numeric values in close: 1

========== DATA CLEAN REPORT ========
Some rows removed, File : 260228_trade-summary-test-dup.csv before cleaning → 293 rows, After 291 rows


============== DATA QUALITY REPORT ==============
FILE: 260228_trade-summary-test.csv
Total rows: 293
Missing symbols: 1
Missing close prices: 1
============== ENF OF DATA QUALITY REPORT ==============

Invalid numeric values in close: 1

========== DATA CLEAN REPORT ========
Some rows removed, File : 260228_trade-summary-test.csv before cleaning → 293 rows, After 291 rows

All CSVs inserted into DuckDB.
Removing duplicates...
Sorting dataset...
Creating composite index on symbol+date for speed...
Ex

### Test speed of query execution with indexing

In [10]:
# commnet above code's indexing part and give a dofferent name for db cvvration try it with below code
# try agian with uncommenting

import time
import duckdb

# Change this to the DB you want to test
con = duckdb.connect("cse_data.db")  # or 'cse_data_index.db'

query = "SELECT * FROM stocks_sorted WHERE symbol='JKH'"

start = time.time()
results = con.execute(query).fetchall()
end = time.time()

print(f"Rows returned: {len(results)}")
print(f"Query time: {end - start:.4f} seconds")

Rows returned: 0
Query time: 0.0068 seconds


In [11]:
df.columns

Index(['company', 'symbol', 'volume', 'trades', 'prev_close', 'open', 'high',
       'low', 'close', 'change', 'change_percentage', 'date'],
      dtype='object')

In [12]:
df = pd.read_csv(r"C:\Users\Lapmart\Jupyter AI Models\CSE Investment Assistence/MASTER_CSE_DATA.csv")
df.head()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
0,ASIA ASSET FINANCE PLC,AAF.N0000,184992.0,102.0,61.7,61.5,63.0,60.8,60.9,-0.8,-1.30,2025-10-23,1
1,ASIA ASSET FINANCE PLC,AAF.N0000,225496.0,128.0,60.9,61.6,62.0,60.0,60.3,-0.6,-0.99,2025-10-24,1
2,ASIA ASSET FINANCE PLC,AAF.N0000,217100.0,171.0,60.3,61.0,61.0,58.5,59.2,-1.1,-1.82,2025-10-27,1
3,ASIA ASSET FINANCE PLC,AAF.N0000,107995.0,93.0,59.2,60.9,61.0,59.0,59.9,0.7,1.18,2025-10-28,1
4,ASIA ASSET FINANCE PLC,AAF.N0000,217448.0,101.0,59.9,60.0,62.0,59.0,60.5,0.6,1.00,2025-10-29,1


In [13]:
df.tail()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
15124,YORK ARCADE HOLDINGS PLC,YORK.N0000,131286.0,143.0,16.8,16.8,17.2,16.5,16.6,-0.2,-1.19,2026-02-24,1
15125,YORK ARCADE HOLDINGS PLC,YORK.N0000,240721.0,251.0,16.6,16.9,16.9,16.2,16.3,-0.3,-1.81,2026-02-25,1
15126,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-26,1
15127,YORK ARCADE HOLDINGS PLC,YORK.N0000,414474.0,215.0,16.1,16.3,17.2,16.0,16.3,0.2,1.24,2026-02-27,1
15128,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-28,1


# **PHASE 2 — FEATURE ENGINEERING ENGINE**
- For 10 day profit
- let create new table **stocks_features** > reads data from table **stocks_sorted** > copies all original columns > Adds new calculated columns
- WINDOW w → defines a reusable window that we can use any where insted of typing long query several times,it say defines how rows are grouped and ordered
  - grouped by stock symbol
  - ordered by date

In [20]:
# ============== Create Feature Table Base ===========================================================

log("Creating feature base table...")

# Opens a connection to a DuckDB database file named cse_data.db, If it doesn’t exist, DuckDB creates it
con = duckdb.connect(database="cse_data.db")

# Creates a table called stocks_features, If it already exists → replaces it
# SELECT *,  > Selects all columns
# LAG window function looks backward in the result set, returns the value of the close column from 5 rows earlier

# close_5d     --> price 5 day ealier 
# close_10d    --> price 10 ddays earlier, 
# high_20      --> highest closing price from the last 20 rows (including today)
# low_20       --> lowest closing price from the last 20 rows (including today)
# avg_vol_20   --> average volume over the last 20 records including today
# std_vol_20   --> standard deviation, how stable or unstable volume has been recently(how much the trading volume has been changing(varying) over the 
#                  last 20 rows including today) if small : volume is stable if large : volume is fluctuating a lot 
#                  how to cal : Find the average of last 20 rec > Find difference from average of each > Square differences(ignore - vals) 
#                  > Average of squared values > Square root 
# std_close_10 --> how much the closing price has been changing (varying) over the last 10 rows including today
con.execute("""
CREATE OR REPLACE TABLE stocks_features AS
SELECT
    *,
    
    LAG(close, 5)  OVER w AS close_5d,
    LAG(close, 10) OVER w AS close_10d,

    MAX(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS high_20,
    MIN(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS low_20,

    AVG(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS avg_vol_20,
    STDDEV(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS std_vol_20,

    STDDEV(close) OVER (w ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) AS std_close_10

FROM stocks_sorted
WINDOW w AS (PARTITION BY symbol ORDER BY date);
""")


# =================== Creating Momentum Features ==========================================================

# Momentum = how fast price is changing
# ret_5  --> Calculates 5-day return as (Current price ÷ price 5 days ago) − 1 >>> Without -1 : 120 / 100 = 1.2, but we need growth  as 0.2, the use -1
# ret_10 --> Calculate 10-day return
log("Creating momentum features...")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_5 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_5 = (close / close_5d) - 1;
""")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_10 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_10 = (close / close_10d) - 1;
""")


# =================== Creating Liquidity Features =========================================================

# how easy it is to buy/sell stock
# volume_ratio --> volume_ratio = volume / avg_vol_20 >>> If >1 → today volume is higher than normal, If <1 → lower than normal 
# volume_z     --> how unusual today’s volume is, measured in units of normal variation
#                  (volume - avg_vol_20) / std_vol_20, 
#                  volume - avg_vol_20 >>> How much today’s volume differs from the recent average.
#                  / std_vol_20 >>> normalizes the difference in terms of “typical variation.”
#                  > 0 → volume higher than normal, < 0 → volume lower than normal, ≈ 0 → volume is typical
log("Creating liquidity features...")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_ratio DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_ratio = volume / avg_vol_20;
""")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_z DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_z = (volume - avg_vol_20) / std_vol_20;
""")


# =================== Creating Breakout Context =========================================================

# range_position >>> where today’s price lies in 20-day range
#                    0 = at 20-day low, 0.5 = middle, 1 = at 20-day high
# detect => breakouts, overbought zones, support/resistance
con.execute("ALTER TABLE stocks_features ADD COLUMN range_position DOUBLE;")

con.execute("""
UPDATE stocks_features
SET range_position = (close - low_20) / (high_20 - low_20);
""")


# =================== Remove Invalid Rows =======================================================================

"""We create a new table so the original data stays safe and unchanged for debugging or reuse.
It also makes the ML pipeline cleaner and reproducible by separating raw data from cleaned data"""
# AT THIS POINT WE CREATE NEW TABLE CALLED "stocks_features_clean"
# Creates a cleaned version of the table
# Removes rows where:
#            close_10d IS NULL
#            avg_vol_20 IS NULL
#            std_vol_20 IS NULL
#            std_close_10 IS NULL
#            volume > 0
#            close > 0
# Why cleaning need ? Because early rows don't have enough past data to calculate rolling features
log("Cleaning feature table...")

con.execute("""
CREATE OR REPLACE TABLE stocks_features_clean AS
SELECT *
FROM stocks_features
WHERE
    close_10d IS NOT NULL
    AND avg_vol_20 IS NOT NULL
    AND std_vol_20 IS NOT NULL
    AND std_close_10 IS NOT NULL
    AND volume > 0
    AND close > 0;
""")


# =================== Liquidity Ranking ==============================================================================

# PARTITION / group stocks by day then order by avg_vol_20 in descending order, Look on a particular date, decide who is #1, #2, etc., based on volume
# Highest volume → rank 1, Equal volumes → same rank, Next rank skips number (because of ties,  > 1, 2, 2, 4),  assign values for each relvent row in the 
# liquidity_rankcolumn in table(stocks_features_clean) by matching symbol with its calculated rank, this doesnt change table row order 
# only theliquidity_rank column, this column is not perfect as 1, 2, 3, may be 3, 1, 2
con.execute("ALTER TABLE stocks_features_clean ADD COLUMN liquidity_rank DOUBLE;")

con.execute("""
UPDATE stocks_features_clean t
SET liquidity_rank = r.rank
FROM (
    SELECT
        symbol,
        date,
        RANK() OVER (PARTITION BY date ORDER BY avg_vol_20 DESC) AS rank
    FROM stocks_features_clean
) r
WHERE t.symbol = r.symbol
AND t.date = r.date;
""")


# =================== Export Feature Dataset ============================================================================
# This is the final csv data file used for ML training
log("Exporting feature dataset...")

con.execute("""
COPY stocks_features_clean
TO 'features_master.csv'
(HEADER, DELIMITER ',');
""")

Creating feature base table...
Creating momentum features...
Creating liquidity features...
Cleaning feature table...
Exporting feature dataset...


### Chekc the main data in tempory table "stocks_features"

In [26]:
# Counts all rows in the table
print("all rows : ", con.execute("SELECT COUNT(*) FROM stocks_features").fetchall())

# Counts how many rows are missing the ret_5 value
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features WHERE ret_5 IS NULL").fetchall())

# Counts how many rows are missing the ret_10 value
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features WHERE ret_10 IS NULL").fetchall())

# Finds earliest and latest date in the table
print("data from-to ", con.execute("SELECT MIN(date), MAX(date) FROM stocks_features").fetchall())

all rows :  [(15129,)]
missing for ret_10 :  [(1506,)]
missing for ret_10 :  [(2992,)]
data from-to  [(datetime.date(2025, 10, 23), datetime.date(2026, 2, 28))]


### Check the main data in table "stocks_features_clean"

In [29]:
# Counts all rows in the table
print("all rows : ", con.execute("SELECT COUNT(*) FROM stocks_features").fetchall())

# Counts how many rows are missing the ret_10 value.
print("missing for ret_5 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean WHERE ret_5 IS NULL").fetchall())

# Counts how many rows are missing the ret_10 value.
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean WHERE ret_10 IS NULL").fetchall())

# Finds earliest and latest date in the table.
print("data from-to ", con.execute("SELECT MIN(date), MAX(date) FROM stocks_features_clean").fetchall())

all rows :  [(15129,)]
missing for ret_5 :  [(0,)]
missing for ret_10 :  [(0,)]
data from-to  [(datetime.date(2025, 11, 7), datetime.date(2026, 2, 28))]


### Check table content

In [34]:
import pandas as pd

df = con.execute("""
SELECT *
FROM stocks_features
LIMIT 5;
""").fetchdf()

print(df)

                   company     symbol       low  close  prev_close  open  \
0  ALPHA FIRE SERVICES PLC  AFS.N0000  182884.0   62.0        27.6  27.6   
1  ALPHA FIRE SERVICES PLC  AFS.N0000   39690.0   40.0        27.3  27.0   
2  ALPHA FIRE SERVICES PLC  AFS.N0000   43605.0   26.0        26.4  26.5   
3  ALPHA FIRE SERVICES PLC  AFS.N0000   46556.0   39.0        27.0  27.0   
4  ALPHA FIRE SERVICES PLC  AFS.N0000   12473.0   29.0        26.5  27.9   

   high  volume  trades  change  ...  high_20 low_20  avg_vol_20  std_vol_20  \
0  28.7    27.0    27.3    -0.3  ...     62.0   62.0      27.000         NaN   
1  27.3    26.0    26.4    -0.9  ...     62.0   40.0      26.500    0.707107   
2  27.6    26.5    27.0     0.6  ...     62.0   26.0      26.500    0.500000   
3  28.0    26.4    26.5    -0.5  ...     62.0   26.0      26.475    0.411299   
4  27.9    26.0    27.1     0.6  ...     62.0   26.0      26.380    0.414729   

   std_close_10  ret_5  ret_10  volume_ratio  volume_z  range_

In [40]:
df = con.execute("""
SELECT *
FROM stocks_features_clean
LIMIT 5;
""").fetchdf()

print(df)

                   company     symbol      low  close  prev_close  open  high  \
0  ALPHA FIRE SERVICES PLC  AFS.N0000   6900.0   13.0        26.5  26.0  27.8   
1  ALPHA FIRE SERVICES PLC  AFS.N0000  29247.0   29.0        26.9  27.0  27.0   
2  ALPHA FIRE SERVICES PLC  AFS.N0000  24239.0   26.0        26.8  27.0  28.9   
3  ALPHA FIRE SERVICES PLC  AFS.N0000   8632.0   14.0        27.7  27.4  27.5   
4  ALPHA FIRE SERVICES PLC  AFS.N0000   2676.0    6.0        27.0  26.9  27.5   

   volume  trades  change  ...  low_20 avg_vol_20  std_vol_20  std_close_10  \
0    26.0    26.9     0.4  ...    13.0  26.690909    1.056839     27.133620   
1    25.1    26.6    -0.3  ...    13.0  26.558333    1.107379     27.369082   
2    26.9    27.7     0.9  ...    13.0  26.584615    1.064461     27.369082   
3    27.0    27.0    -0.7  ...    13.0  26.614286    1.028709     28.507504   
4    26.9    27.0     0.0  ...     6.0  26.633333    0.994030     30.082110   

      ret_5    ret_10  volume_ratio  v